# 3.3 — Retreino com Inputs Reduzidos: ANN (TensorFlow/Keras)

Fase 3 da Etapa 3: retreina a ANN para k ∈ {4, 5, 6} inputs e os 3 outputs.  
Arquitetura idêntica à Etapa 2.2 — 2×8 ReLU, Adam, MSE, 500 epochs (D-E3-06).  
Total: 9 runs no experimento `reduzido`.

## Seção 1 — Imports e configuração

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

RANDOM_STATE  = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

BASE_DIR      = Path("/Users/lorenzoferreira/Documents/UFRGS/TCC_SBO")
PROCESSED_DIR = BASE_DIR / "ARTEFATOS" / "ETAPA_0" / "processed"
SELECAO_DIR   = BASE_DIR / "ARTEFATOS" / "ETAPA_3" / "selecao"
ARTIFACTS_DIR = BASE_DIR / "ARTEFATOS" / "ETAPA_3" / "reduzido"
MLFLOW_URI    = f"file:///{BASE_DIR}/mlruns"
EXPERIMENT    = "reduzido"

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)
print("TensorFlow:", tf.__version__)
print("MLflow URI:", mlflow.get_tracking_uri())
print("Experimento:", EXPERIMENT)

## Seção 2 — Carga dos dados e subconjuntos de features

In [ ]:
X_train = np.load(PROCESSED_DIR / "X_train.npy")
X_val   = np.load(PROCESSED_DIR / "X_val.npy")
X_test  = np.load(PROCESSED_DIR / "X_test.npy")
y_train = np.load(PROCESSED_DIR / "y_train.npy")
y_val   = np.load(PROCESSED_DIR / "y_val.npy")
y_test  = np.load(PROCESSED_DIR / "y_test.npy")

scaler_y_min   = np.load(PROCESSED_DIR / "scaler_y_min.npy")
scaler_y_scale = np.load(PROCESSED_DIR / "scaler_y_scale.npy")

FEATURE_NAMES = ["P1", "T1", "T2", "RRC1", "BRC1", "RRC2", "BRC2", "RFF"]
FEAT_IDX      = {f: i for i, f in enumerate(FEATURE_NAMES)}
OUTPUT_NAMES  = ["M_CH3OH", "x_CH3OH", "ET"]

subsets_df = pd.read_csv(SELECAO_DIR / "3.2_subconjuntos.csv")
subsets = {}
for _, row in subsets_df.iterrows():
    k = int(row["k"])
    feats = [f.strip() for f in row["features"].split(",")]
    subsets[k] = feats

def denormalize_y(y_norm, idx):
    return y_norm * scaler_y_scale[idx] + scaler_y_min[idx]

print("X_train:", X_train.shape, "| X_val:", X_val.shape, "| X_test:", X_test.shape)
for k, feats in sorted(subsets.items()):
    print(f"  S_{k}: {feats}")

## Seção 2 — Definição da arquitetura

In [ ]:
def build_model(input_dim):
    """Arquitetura idêntica à Etapa 2.2 — única diferença é input_dim variável (D-E3-06)."""
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(8, activation="relu"),
        layers.Dense(8, activation="relu"),
        layers.Dense(1),
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

# Verificar arquitetura para k=4 (menor)
build_model(input_dim=4).summary()

## Seção 3 — Loop de treinamento (k × output)

In [ ]:
histories = {}
results   = []

for k in [4, 5, 6]:
    feat_list = subsets[k]
    col_idx   = [FEAT_IDX[f] for f in feat_list]

    X_tr_k  = X_train[:, col_idx]
    X_val_k = X_val[:,   col_idx]
    X_te_k  = X_test[:,  col_idx]

    for output_idx, output_name in enumerate(OUTPUT_NAMES):
        print(f"\n{'='*55}")
        print(f"  ANN  |  k={k}  |  {output_name}")
        print(f"{'='*55}")

        # Reset seed por output (reprodutibilidade)
        np.random.seed(RANDOM_STATE)
        tf.random.set_seed(RANDOM_STATE)

        y_tr = y_train[:, output_idx]
        y_vl = y_val[:,   output_idx]
        y_te = y_test[:,  output_idx]

        model = build_model(input_dim=k)

        history = model.fit(
            X_tr_k, y_tr,
            validation_data=(X_val_k, y_vl),
            epochs=500,
            batch_size=32,
            verbose=0,
        )
        histories[(k, output_name)] = history
        print(f"  Epochs treinados: {len(history.history['loss'])}")

        p_tr_norm  = model.predict(X_tr_k,  verbose=0).flatten()
        p_val_norm = model.predict(X_val_k, verbose=0).flatten()
        p_te_norm  = model.predict(X_te_k,  verbose=0).flatten()

        y_tr_real  = denormalize_y(y_tr, output_idx)
        y_vl_real  = denormalize_y(y_vl, output_idx)
        y_te_real  = denormalize_y(y_te, output_idx)
        p_tr_real  = denormalize_y(p_tr_norm, output_idx)
        p_vl_real  = denormalize_y(p_val_norm, output_idx)
        p_te_real  = denormalize_y(p_te_norm, output_idx)

        metrics = {}
        for split, y_real, p_real in [
            ("train", y_tr_real, p_tr_real),
            ("val",   y_vl_real, p_vl_real),
            ("test",  y_te_real, p_te_real),
        ]:
            metrics[f"{split}_r2"]  = r2_score(y_real, p_real)
            metrics[f"{split}_mae"] = mean_absolute_error(y_real, p_real)
            metrics[f"{split}_mse"] = mean_squared_error(y_real, p_real)
            print(f"  {split:5s} → R²={metrics[f'{split}_r2']:.4f}  MAE={metrics[f'{split}_mae']:.4f}")

        # Artefato .keras
        art_path = ARTIFACTS_DIR / "ANN" / output_name / f"k{k}"
        art_path.mkdir(parents=True, exist_ok=True)
        model_path = art_path / "model.keras"
        model.save(str(model_path))

        # Run MLflow
        with mlflow.start_run(run_name=f"ANN_{output_name}_k{k}"):
            mlflow.set_tag("model",  "ANN")
            mlflow.set_tag("output", output_name)
            mlflow.set_tag("k",      str(k))
            mlflow.set_tag("fase",   "retreino")

            mlflow.log_params({
                "arquitetura":         "ANN",
                "output":              output_name,
                "k":                   k,
                "inputs_selecionados": str(feat_list),
                "metodo_selecao":      "shap_global_topk",
                "random_state":        RANDOM_STATE,
                "n_layers":            2,
                "n_neurons":           8,
                "activation":          "relu",
                "optimizer":           "adam",
                "loss":                "mse",
                "epochs":              500,
                "batch_size":          32,
                "early_stopping":      "none",
            })
            mlflow.log_metrics(metrics)
            mlflow.log_artifact(str(model_path))

        results.append({
            "model":              "ANN",
            "output":             output_name,
            "k":                  k,
            "inputs_selecionados": feat_list,
            **metrics,
        })

print("\n=== Treinamento ANN concluído ===")
print(f"Total de runs: {len(results)} (esperado: 9)")

## Seção 4 — Curvas de aprendizado

In [ ]:
fig_dir = ARTIFACTS_DIR

for k in [4, 5, 6]:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, output_name in zip(axes, OUTPUT_NAMES):
        hist = histories[(k, output_name)].history
        ax.plot(hist["loss"],     label="train")
        ax.plot(hist["val_loss"], label="val", linestyle="--")
        ax.set_title(f"ANN — {output_name} (k={k})")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("MSE (normalizado)")
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.suptitle(f"Curvas de aprendizado — ANN Reduzida (k={k})", fontsize=13, y=1.02)
    plt.tight_layout()

    fig_path = fig_dir / f"3.3_ann_learning_curves_k{k}.png"
    plt.savefig(str(fig_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Salvo: {fig_path}")

## Seção 5 — Tabela de resultados e validação

In [ ]:
results_df = pd.DataFrame(results)

print("=== R² test set — ANN Reduzida (por k e output) ===")
pivot = results_df.pivot_table(
    index="k",
    columns="output",
    values="test_r2",
).round(4)
display(pivot)

print("\n=== Tabela completa ===")
display(
    results_df[[
        "k", "output",
        "train_r2", "val_r2", "test_r2",
        "train_mae", "val_mae", "test_mae",
    ]]
    .sort_values(["k", "output"])
    .reset_index(drop=True)
    .round(4)
)

# Validação
assert len(results_df) == 9, f"Esperado 9 runs, obtido {len(results_df)}"
assert results_df["test_r2"].notna().all(), "NaN encontrado em test_r2"
neg = results_df[results_df["test_r2"] < 0]
if len(neg) > 0:
    print("AVISO: runs com R² negativo:", neg[["k", "output", "test_r2"]].to_dict("records"))
else:
    print("\nValidação OK: 9 runs, nenhum R² NaN ou negativo.")